# Lab 3 · When memory gets in the way of the answer

**About 25 minutes. Starts cold:** the first cell builds the memory it needs (about two minutes of extraction).

Retrieval quality isn't an API call. It's judgment. You'll ask one user's memory three questions and see a clean hit, a near miss, and a question whose answer **isn't in the top 10 at all**. Then you'll find out why, fix it, and ask again.

In [ ]:
import labkit

pool, memory = labkit.connect(source="replay")
USER = labkit.lab_user(3)
labkit.reset_user(memory, USER)
# Two conversations with the same user: an onboarding request and a support ticket
# in which the user corrects the bucket region.
for name in ("onboarding_01", "support_01"):
    _ = labkit.replay(memory, USER, name)
labkit.show_memories(pool, USER)

## 1. Three questions

In [ ]:
def ask(question, highlight=None, **search):
    results = memory.search(question, user_id=USER, max_results=10, **{"record_types": labkit.MEMORY_TYPES, **search})
    print(f"\n{question}")
    labkit.show_results(results, highlight=highlight)
    return results

def rank_of(results, pattern):
    import re
    return next((i for i, r in enumerate(results, 1) if re.search(pattern, r.record.content or "", re.I)), None)

clean = ask("What is the ARN of the IAM role the export job uses?", highlight="arn:")
near = ask("When does the new analysts' access run out?", highlight="90 days")
MIXED = "Which region is my export bucket in, and how should you contact me?"
mixed = ask(MIXED, highlight=labkit.CONTACT_PREFERENCE)
print("\ncontact preference is at rank:", rank_of(mixed, labkit.CONTACT_PREFERENCE) or "not in the top 10")

The first question is specific, and its answer comes first. The second is close, but the answer may share the top with near-copies of itself. The third asks about **two** things, and one topic crowds out the other.

**What filled the slots?** Look for the same fact stored more than once, a stale region next to its correction, and "the assistant asked…" memories.

## 2. What `record_types` does

Without `record_types`, search ranks raw conversation messages alongside durable memories:

In [ ]:
_ = ask(MIXED, highlight=labkit.CONTACT_PREFERENCE, record_types=["memory", "fact", "preference", "guideline", "message"])

Raw messages compete for the same slots. Most agents want memories only, so the labs always pass `record_types`.

## 3. Make it a recorded turn

The health check can only say what reached the prompt if the agent says so. `record_prompt` tells the inspector which results went in; here, the top 5.

In [ ]:
thread = memory.create_thread(user_id=USER, agent_id="support_bot")

def turn(question, reply):
    results = memory.search(question, user_id=USER, record_types=labkit.MEMORY_TYPES, max_results=10)
    used = results[:5]
    memory.inspector.record_prompt(prompt="\n".join(r.record.content for r in used), reply=reply,
                                   memory_ids_used=[r.id for r in used], reply_source="scripted")
    thread.add_messages([{"role": "user", "content": question}, {"role": "assistant", "content": reply}])
    return results

_ = turn(MIXED, "Let me check your bucket region and how you like to be contacted.")
print("why view:", labkit.run_url(thread.thread_id, turn=1, view="why"))
report = labkit.check(pool, USER)

Look for **`crowded_turn`**: it counts how many of the 5 prompt slots went to memories that shouldn't be there. Open the **why** link to see each one badged *stale*, *duplicate* or *transient*.

## 4. Fix what the check found

The check names every memory a fix would remove:
- the stale side of each supersession
- the extra copies of each duplicate
- each transient memory

**Read the list before you delete anything.** A stale memory can hold a detail its replacement lacks, like an expiry date. If you want to keep one, remove it from `to_delete`, or fold its detail into the current memory with `memory.update_memory(...)` first.

In [ ]:
to_delete = {}
for f in report.findings:
    if f.kind == "superseded":
        to_delete[f.evidence["stale"]] = "stale"
    elif f.kind == "transient":
        to_delete[f.memory_ids[0]] = "transient"
    elif f.kind == "duplicate":
        to_delete.update({i: "duplicate" for i in f.memory_ids if i != f.evidence["keep"]})

content = {m["id"]: m["content"] for m in labkit.memories(pool, USER)}
for memory_id, why in to_delete.items():
    print(f"{why:<10} {labkit.shorten(content.get(memory_id, '?'), 110)}")

In [ ]:
for memory_id in to_delete:
    memory.delete_memory(memory_id)
print(f"deleted {len(to_delete)} memories")

## 5. Ask again

In [ ]:
mixed_after = ask(MIXED, highlight=labkit.CONTACT_PREFERENCE)
print("\ncontact preference rank: before", rank_of(mixed, labkit.CONTACT_PREFERENCE) or "not in top 10",
      "→ after", rank_of(mixed_after, labkit.CONTACT_PREFERENCE) or "not in top 10")
_ = turn(MIXED, "Your bucket is in us-west-2, and I'll contact you by email only.")
report_after = labkit.check(pool, USER)

The findings you fixed should show as **resolved**, including the crowded turn. The contact preference should have moved up, but probably not into the top 5. Cleanup gave back the slots that junk was taking, but this question is still mostly about the export bucket, and its embedding says so.

Extraction varies from run to run, so your numbers will differ. What doesn't vary is the method: when a reply is wrong, look at what reached the prompt before blaming the model.


## 6. Ask one thing at a time

A question with two topics gets one embedding, which lands between them. An agent can split it and search each part. Most of the time, that's a better fix than any amount of cleanup:

In [ ]:
region = ask("Which region is my export bucket in?", highlight="us-west-2")
contact = ask("How should you contact me?", highlight=labkit.CONTACT_PREFERENCE)
print("\ncontact preference rank when asked on its own:", rank_of(contact, labkit.CONTACT_PREFERENCE) or "not in top 10")

Asked on its own, the preference should come back at or near the top. Retrieval quality is partly the store (what's in it, and how much of it is junk) and partly the question (what you ask it). This lab changed both.

In [ ]:
memory.close()
pool.close()

## What you saw

- **Specific questions retrieve well. Mixed questions don't.** Duplicates and stale facts take slots, and a real preference can drop out of the top 10. Splitting the question often helps more than cleanup.
- `record_types` keeps raw messages out of the results.
- `record_prompt` lets the inspector judge what actually reached the model: the **crowded turn**.
- The fix loop: **check → read → delete (or merge) → ask again → check.** In this package, cleanup is the agent's job, because extraction never revises.

The dashboard has all of it: the conversation turn by turn, **why** for every reply, the **lifecycle** of each turn, and the **Memory** page's findings.